In [1]:
import ee
import geemap
import json
import geopandas as gpd
import requests

In [2]:
path_to_geojson = "../__geojson__/darney_3_regions.geojson"

In [3]:
ee.Initialize()

with open(path_to_geojson) as f:
    geojson_data = json.load(f)

roi = ee.FeatureCollection(geojson_data)

# Load and process Sentinel-1
s1 = (ee.ImageCollection("COPERNICUS/S1_GRD")
    .filterBounds(roi)
    .filterDate("2024-06-01", "2024-06-30")
    .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
    .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
    .filter(ee.Filter.eq("instrumentMode", "IW")))

def add_s1_ratio(image):
    vv_lin = ee.Image(10).pow(image.select("VV").divide(10))
    vh_lin = ee.Image(10).pow(image.select("VH").divide(10))
    ratio = vv_lin.divide(vh_lin).rename("VV_VH_ratio")
    return image.addBands(ratio).clip(roi).copyProperties(image, ["system:time_start"])

clipped_s1 = s1.map(add_s1_ratio)

# Cloud masking function for Sentinel-2
def mask_s2_clouds(image):
    qa = image.select('QA60')
    # Bit 10 = Opaque clouds, Bit 11 = Cirrus clouds
    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11
    # Both flags must be zero to indicate clear conditions
    mask = qa.bitwiseAnd(cloud_bit_mask).eq(0).And(qa.bitwiseAnd(cirrus_bit_mask).eq(0))
    return image.updateMask(mask)

# Load and process Sentinel-2 (with cloud filtering applied to create a composite image)
s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(roi)
    .filterDate("2024-06-01", "2024-06-30")
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 30))
    .map(mask_s2_clouds))

def add_s2_indices(image):
    ndvi = image.normalizedDifference(["B8", "B4"]).rename("NDVI")
    ndre = image.normalizedDifference(["B8", "B5"]).rename("NDRE")
    return image.addBands([ndvi, ndre]).clip(roi).copyProperties(image, ["system:time_start"])

clipped_s2 = s2.map(add_s2_indices)

# Create temporal mean composites over the period (cloudy pixels are ignored in the mean)
s1_mean = clipped_s1.select(["VV", "VH", "VV_VH_ratio"]).mean()
s2_mean = clipped_s2.select(["NDVI", "NDRE"]).mean()
s2_rgb = clipped_s2.select(["B4", "B3", "B2"]).mean() # Red, Green, Blue

## Layers from cartes.gouv.fr (WFS)

In [4]:
# Bounding box
gdf = gpd.read_file(path_to_geojson)
total_bounds = gdf.total_bounds
bbox_str = f"{total_bounds[0]},{total_bounds[1]},{total_bounds[2]},{total_bounds[3]},EPSG:4326"

# In-memory container for all layers
ign_stack = {}

# Fetch vector layers
wfs_url = "https://data.geopf.fr/wfs/ows"
vector_layers = {
    "courbes_niveau": "ELEVATION.CONTOUR.LINE:courbe",
    "parcelles_cadastrales": "CADASTRALPARCELS.PARCELLAIRE_EXPRESS:parcelle"
}

print("Fetching vector layers into memory...")
for key, typename in vector_layers.items():
    response = requests.get(wfs_url, params={
        "service": "WFS", 
        "version": "2.0.0", 
        "request": "GetFeature",
        "typename": typename, 
        "outputFormat": "application/json",
        "srsName": "EPSG:4326", 
        "bbox": bbox_str
    })
    if response.status_code == 200:
        ign_stack[key] = gpd.GeoDataFrame.from_features(response.json(), crs="EPSG:4326")
        print(f" -> Loaded vector layer: {key} ({len(ign_stack[key])} features)")
    else:
        print(f" -> Failed to fetch vector layer {key}: Status {response.status_code}")

# Configure raster streams (using IGN's reliable WMS-raster endpoint)
wms_raster_url = "https://data.geopf.fr/wms-r/wms"

raster_layers = {
    "plan_ign": ("GEOGRAPHICALGRIDSYSTEMS.PLANIGNV2", "image/png"),
    "scan_25": ("GEOGRAPHICALGRIDSYSTEMS.MAPS.SCAN25", "image/jpeg"),
    "carte_pentes": ("ELEVATION.SLOPES", "image/png"),
    "photo_1950_1965": ("ORTHOIMAGERY.ORTHOPHOTOS.1950-1965", "image/jpeg"),
    "photo_1965_1980": ("ORTHOIMAGERY.ORTHOPHOTOS.1965-1980", "image/jpeg"),
    "orthophotos": ("ORTHOIMAGERY.ORTHOPHOTOS", "image/jpeg"),
    "carte_sols": ("PEDOLOGY.SOILMAPS", "image/png"),
    "mnt_lidar_hd": ("IGNF_LIDAR-HD_MNT_ELEVATION.ELEVATIONGRIDCOVERAGE", "image/tiff"),
    "forets_publiques": ("ONF:FORET_PUBLIQUE", "image/png")
}

print("\nConfiguring raster stream layers in memory...")
count = 3
for key, (layer_name, img_format) in raster_layers.items():
    ign_stack[key] = {
        "type": "WMS",
        "url": wms_raster_url,
        "layer": layer_name,
        "format": img_format
    }
    print(f" -> [{count}] Registered raster stream: {key}")
    count += 1

Fetching vector layers into memory...
 -> Loaded vector layer: courbes_niveau (80 features)
 -> Loaded vector layer: parcelles_cadastrales (4795 features)

Configuring raster stream layers in memory...
 -> [3] Registered raster stream: plan_ign
 -> [4] Registered raster stream: scan_25
 -> [5] Registered raster stream: carte_pentes
 -> [6] Registered raster stream: photo_1950_1965
 -> [7] Registered raster stream: photo_1965_1980
 -> [8] Registered raster stream: orthophotos
 -> [9] Registered raster stream: carte_sols
 -> [10] Registered raster stream: mnt_lidar_hd
 -> [11] Registered raster stream: forets_publiques


## Google Earth Engine interactive map

In [5]:
m = geemap.Map()
m.centerObject(roi, 12)

# Sentinel layers
m.addLayer(s2_rgb, {"min": 0, "max": 3000}, "Sentinel-2 True Color (RGB)")

ndvi_vis = {"min": 0.2, "max": 0.8, "palette": ["brown", "yellow", "lightgreen", "darkgreen"]}
m.addLayer(s2_mean.select("NDVI"), ndvi_vis, "Sentinel-2 NDVI")
m.addLayer(s1_mean.select("VV"), {"min": -25, "max": -5}, "Sentinel-1 VV (dB)")

# IGN layers
layer_titles = {
    "plan_ign": "Plan IGN",
    "scan_25": "SCAN 25 Carte topographique",
    "carte_pentes": "Carte des pentes",
    "photo_1950_1965": "Aerial Photo (1950-1965)",
    "photo_1965_1980": "Aerial Photo (1965-1980)",
    "orthophotos": "Orthophotos",
    "carte_sols": "Carte des sols",
    "mnt_lidar_hd": "LiDAR HD MNT (Elevation)",
    "forets_publiques": "Forêts publiques (ONF)",
    "courbes_niveau": "Courbes de niveau",
    "parcelles_cadastrales": "Parcelles cadastrales"
}

for key, item in ign_stack.items():
    title = layer_titles.get(key, key)
    
    # Vector layers
    if isinstance(item, gpd.GeoDataFrame):
        style = {"color": "brown", "fillColor": "none", "weight": 0.5} if key == "courbes_niveau" else {"color": "yellow", "fillColor": "none", "weight": 1}
        m.add_gdf(item, layer_name=title, style=style)
        
    # Raster layers (WMS)
    elif isinstance(item, dict) and item.get("type") == "WMS":
        m.add_wms_layer(
            url=item["url"],
            layers=item["layer"],
            name=title,
            format=item["format"],
            transparent=True,
            opacity=0.8,
            shown=False
        )

# Polygons
m.addLayer(roi, {"color": "FF8C00", "fillColor": "00000000"}, "Darney Polygons")

m.addLayerControl()
m

Map(center=[48.08676739878947, 6.059427708389103], controls=(WidgetControl(options=['position', 'transparent_b…